In [ ]:
#!/usr/bin/env python3
"""
extract_radial_images.py

Extracts camera images from each sequence in the Radial Dataset using the SyncReader API.
"""
import os
import glob
import argparse
from DBReader import SyncReader
import cv2
from tqdm import tqdm
import numpy as np

def save_images_from_sequence(db, seq_dir: str, output_root: str) -> None:
    """
    Reads all camera frames from a single sequence and saves them as PNG files.

    Filenames will be of the form:
        {sequence_name}_{frame_index:06d}.png

    Args:
        seq_dir: Path to the sequence folder (e.g., ".../RECORD@2020-11-22_12.42.37").
        output_root: Directory under which a subfolder for this sequence will be created.
    """
    seq_name = os.path.basename(seq_dir.rstrip(os.sep))

    num_samples = len(db)

    # Create output directory for this sequence's camera images
    out_dir = os.path.join(output_root, 'images')
    os.makedirs(out_dir, exist_ok=True)

    # Iterate and save
    for idx in tqdm(range(num_samples), desc=f"Image Seq {seq_name}"):
        try:
            data = db.GetSensorData(idx)
        except Exception as e:
            print(f"Index error in sequence {seq_name}: {e}, skipping.")
            continue
        img = data['camera']['data']  # assumed HxW or HxWx3 numpy array

        # If RGB, convert to BGR for OpenCV
        # if img.ndim == 3 and img.shape[2] == 3:
        #     img_out = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        # else:
        #     img_out = img
        img_out = img

        filename = f"{seq_name}_{idx:06d}.png"
        cv2.imwrite(os.path.join(out_dir, filename), img_out)

def save_radarcube_from_sequence(db, seq_dir: str, output_root: str) -> None:

    def get_radar_cube(sample):

        numSamplePerChirp = 512
        numRxPerChip = 4
        numChirps = 256
        numRxAnt = 16
        numTxAnt = 12
        numReducedDoppler = 16
        numChirpsPerLoop = 16
        
        adc0 = sample['radar_ch0']['data']
        adc1 = sample['radar_ch1']['data']
        adc2 = sample['radar_ch2']['data']
        adc3 = sample['radar_ch3']['data']

        frame0 = np.reshape(adc0[0::2] + 1j*adc0[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        frame1 = np.reshape(adc1[0::2] + 1j*adc1[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        frame2 = np.reshape(adc2[0::2] + 1j*adc2[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        frame3 = np.reshape(adc3[0::2] + 1j*adc3[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        radar_frame = np.concatenate([frame3,frame0,frame1,frame2],axis=2)

        # radar_frame_mag = np.abs(radar_frame)
        # radar_frame_norm = radar_frame / (2**12) # np.std(radar_frame_mag)

        # frame_ri = np.concatenate([radar_frame_norm.real, radar_frame_norm.imag], axis=2) 

        return radar_frame

    def get_radar_adc(sample):
        adc0 = sample['radar_ch0']['data']
        adc1 = sample['radar_ch1']['data']
        adc2 = sample['radar_ch2']['data']
        adc3 = sample['radar_ch3']['data']

        return np.stack([adc0, adc1, adc2, adc3], axis=1)

    seq_name = os.path.basename(seq_dir.rstrip(os.sep))

    num_samples = len(db)

    # Create output directory for this sequence's radar cubes
    out_dir = os.path.join(output_root, 'radar_cubes')
    os.makedirs(out_dir, exist_ok=True)

    # Iterate and save
    for idx in tqdm(range(num_samples), desc=f"Radar Seq {seq_name}"):
        try:
            data = db.GetSensorData(idx)
        except Exception as e:
            print(f"Index error in sequence {seq_name}: {e}, skipping.")
            continue
        # radar_cube = get_radar_cube(data) ## Complex radar cube
        radar_adc = get_radar_adc(data)

        filename = f"{seq_name}_{idx:06d}.npy"
        np.save(os.path.join(out_dir, filename), radar_adc)


def save_lidar_from_sequence(db, seq_dir: str, output_root: str) -> None:

    def ConpensateLayerAngle(pcl,index,sensor_height):
        
        offset=0
        if(index%2==0):
            offset = np.deg2rad(.6)

        x = pcl[:,4] * np.cos(pcl[:,5]+offset) * np.cos(pcl[:,6])
        y = pcl[:,4] * np.cos(pcl[:,5]+offset) * np.sin(pcl[:,6])
        z = pcl[:,4] * np.sin(pcl[:,5]+offset) + sensor_height
        
        pcl[:,0] = x
        pcl[:,1] = y
        pcl[:,2] = z
        
        return pcl
    
    def get_pointcloud(sample):
        pts = sample['scala']['data']
        pts = ConpensateLayerAngle(pts, sample['scala']['sample_number'], 0.42)
        return pts
    
    seq_name = os.path.basename(seq_dir.rstrip(os.sep))

    num_samples = len(db)

    # Create output directory for this sequence's radar cubes
    out_dir = os.path.join(output_root, 'pointcloud')
    os.makedirs(out_dir, exist_ok=True)

    # Iterate and save
    for idx in tqdm(range(num_samples), desc=f"Lidar Seq {seq_name}"):

        try:
            data = db.GetSensorData(idx)
        except Exception as e:
            print(f"Index error in sequence {seq_name}: {e}, skipping.")
            continue
        pts = get_pointcloud(data) ## Complex radar cube

        filename = f"{seq_name}_{idx:06d}.npy"
        np.save(os.path.join(out_dir, filename), pts)



def process_all_sequences(root_dir: str, output_root: str) -> None:
    """
    Finds all RECORD@* subdirectories under `root_dir` and extracts camera images.
    """
    pattern = os.path.join(root_dir, 'RECORD@*')
    seq_dirs = sorted(d for d in glob.glob(pattern) if os.path.isdir(d))

    print(f"Total sequences {len(seq_dirs)}")
    file_no = 0
    for seq in seq_dirs:
        print(f"Extracting sequence no {file_no}")
        file_no+=1
        seq_name = os.path.basename(seq.rstrip(os.sep))
        imgs_done = glob.glob(os.path.join(output_root, 'images', f"{seq_name}_*.png"))
        radar_done = glob.glob(os.path.join(output_root, 'radar_cubes', f"{seq_name}_*.npy"))
        lidar_done = glob.glob(os.path.join(output_root, 'pointcloud', f"{seq_name}_*.npy"))
        if (len(imgs_done) and len(radar_done) and len(lidar_done)):
            print("Already extracted, skipping")
        else:
            db = SyncReader(seq, silent=True)
            if (len(imgs_done)==0):
                save_images_from_sequence(db, seq, output_root)
            if (len(radar_done)==0):
                save_radarcube_from_sequence(db, seq, output_root)
            if (len(lidar_done)==0):
                save_lidar_from_sequence(db, seq, output_root)
            

input_root = 'data_raw_sequences'
output_root = 'raw_sequences_extracted'

process_all_sequences(input_root, output_root)


In [ ]:
import numpy as np

final = np.load('scripts/generate_RADIal_from_raw_sequences/label_candidates_0deg.npy',allow_pickle=True)

print(final)

### Making sure synchronization is correct

In [ ]:
#!/usr/bin/env python3
"""
extract_radial_images.py

Extracts camera images from each sequence in the Radial Dataset using the SyncReader API.
"""
import os
import glob
import argparse
from DBReader import SyncReader
import cv2
from tqdm import tqdm
import numpy as np

def save_images_from_sequence(db, seq_dir: str, output_root: str) -> None:
    """
    Reads all camera frames from a single sequence and saves them as PNG files.

    Filenames will be of the form:
        {sequence_name}_{frame_index:06d}.png

    Args:
        seq_dir: Path to the sequence folder (e.g., ".../RECORD@2020-11-22_12.42.37").
        output_root: Directory under which a subfolder for this sequence will be created.
    """
    seq_name = os.path.basename(seq_dir.rstrip(os.sep))

    num_samples = len(db)

    # Create output directory for this sequence's camera images
    out_dir = os.path.join(output_root, 'images')
    os.makedirs(out_dir, exist_ok=True)

    # Iterate and save
    for idx in tqdm(range(num_samples), desc=f"Image Seq {seq_name}"):
        try:
            data = db.GetSensorData(idx)
        except Exception as e:
            print(f"Index error in sequence {seq_name}: {e}, skipping.")
            continue
        img = data['camera']['data']  # assumed HxW or HxWx3 numpy array

        # If RGB, convert to BGR for OpenCV
        # if img.ndim == 3 and img.shape[2] == 3:
        #     img_out = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        # else:
        #     img_out = img
        img_out = img

        filename = f"{seq_name}_{idx:06d}.png"
        cv2.imwrite(os.path.join(out_dir, filename), img_out)

def save_radarcube_from_sequence(db, seq_dir: str, output_root: str) -> None:

    def get_radar_cube(sample):

        numSamplePerChirp = 512
        numRxPerChip = 4
        numChirps = 256
        numRxAnt = 16
        numTxAnt = 12
        numReducedDoppler = 16
        numChirpsPerLoop = 16
        
        adc0 = sample['radar_ch0']['data']
        adc1 = sample['radar_ch1']['data']
        adc2 = sample['radar_ch2']['data']
        adc3 = sample['radar_ch3']['data']

        frame0 = np.reshape(adc0[0::2] + 1j*adc0[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        frame1 = np.reshape(adc1[0::2] + 1j*adc1[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        frame2 = np.reshape(adc2[0::2] + 1j*adc2[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        frame3 = np.reshape(adc3[0::2] + 1j*adc3[1::2], (numSamplePerChirp,numRxPerChip, numChirps), order ='F').transpose((0,2,1))   
        radar_frame = np.concatenate([frame3,frame0,frame1,frame2],axis=2)

        # radar_frame_mag = np.abs(radar_frame)
        # radar_frame_norm = radar_frame / (2**12) # np.std(radar_frame_mag)

        # frame_ri = np.concatenate([radar_frame_norm.real, radar_frame_norm.imag], axis=2) 

        return radar_frame

    def get_radar_adc(sample):
        adc0 = sample['radar_ch0']['data']
        adc1 = sample['radar_ch1']['data']
        adc2 = sample['radar_ch2']['data']
        adc3 = sample['radar_ch3']['data']

        return np.stack([adc0, adc1, adc2, adc3], axis=1)

    seq_name = os.path.basename(seq_dir.rstrip(os.sep))

    num_samples = len(db)

    # Create output directory for this sequence's radar cubes
    out_dir = os.path.join(output_root, 'radar_cubes')
    os.makedirs(out_dir, exist_ok=True)

    # Iterate and save
    for idx in tqdm(range(num_samples), desc=f"Radar Seq {seq_name}"):
        try:
            data = db.GetSensorData(idx)
        except Exception as e:
            print(f"Index error in sequence {seq_name}: {e}, skipping.")
            continue
        # radar_cube = get_radar_cube(data) ## Complex radar cube
        radar_adc = get_radar_adc(data)

        filename = f"{seq_name}_{idx:06d}.npy"
        np.save(os.path.join(out_dir, filename), radar_adc)


def save_lidar_from_sequence(db, seq_dir: str, output_root: str) -> None:

    def ConpensateLayerAngle(pcl,index,sensor_height):
        
        offset=0
        if(index%2==0):
            offset = np.deg2rad(.6)

        x = pcl[:,4] * np.cos(pcl[:,5]+offset) * np.cos(pcl[:,6])
        y = pcl[:,4] * np.cos(pcl[:,5]+offset) * np.sin(pcl[:,6])
        z = pcl[:,4] * np.sin(pcl[:,5]+offset) + sensor_height
        
        pcl[:,0] = x
        pcl[:,1] = y
        pcl[:,2] = z
        
        return pcl
    
    def get_pointcloud(sample):
        pts = sample['scala']['data']
        pts = ConpensateLayerAngle(pts, sample['scala']['sample_number'], 0.42)
        return pts
    
    seq_name = os.path.basename(seq_dir.rstrip(os.sep))

    num_samples = len(db)

    # Create output directory for this sequence's radar cubes
    out_dir = os.path.join(output_root, 'pointcloud')
    os.makedirs(out_dir, exist_ok=True)

    # Iterate and save
    for idx in tqdm(range(num_samples), desc=f"Lidar Seq {seq_name}"):

        try:
            data = db.GetSensorData(idx)
        except Exception as e:
            print(f"Index error in sequence {seq_name}: {e}, skipping.")
            continue
        pts = get_pointcloud(data) ## Complex radar cube

        filename = f"{seq_name}_{idx:06d}.npy"
        np.save(os.path.join(out_dir, filename), pts)



def process_all_sequences(root_dir: str, output_root: str) -> None:
    """
    Finds all RECORD@* subdirectories under `root_dir` and extracts camera images.
    """
    pattern = os.path.join(root_dir, 'RECORD@*')
    seq_dirs = sorted(d for d in glob.glob(pattern) if os.path.isdir(d))

    print(f"Total sequences {len(seq_dirs)}")
    file_no = 0
    for seq in seq_dirs:
        print(f"Extracting sequence no {file_no}")
        file_no+=1
        seq_name = os.path.basename(seq.rstrip(os.sep))
        imgs_done = glob.glob(os.path.join(output_root, 'images', f"{seq_name}_*.png"))
        radar_done = glob.glob(os.path.join(output_root, 'radar_cubes', f"{seq_name}_*.npy"))
        lidar_done = glob.glob(os.path.join(output_root, 'pointcloud', f"{seq_name}_*.npy"))
        if (len(imgs_done) and len(radar_done) and len(lidar_done)):
            print("Already extracted, skipping")
        else:
            db = SyncReader(seq, silent=True)
            if (len(imgs_done)==0):
                save_images_from_sequence(db, seq, output_root)
            if (len(radar_done)==0):
                save_radarcube_from_sequence(db, seq, output_root)
            if (len(lidar_done)==0):
                save_lidar_from_sequence(db, seq, output_root)
            

input_root = 'data_raw_sequences'
output_root = 'raw_sequences_extracted'

process_all_sequences(input_root, output_root)


In [ ]:
import numpy as np

# enable pickle to restore the dict
labeled_seq = np.load(
    'scripts/generate_RADIal_from_raw_sequences/label_candidates_0deg.npy',
    allow_pickle=True
).item()

labeled_seq = np.load(
    'scripts/generate_RADIal_from_raw_sequences/fp_frames.npy',
    allow_pickle=True
).item()


# gives you a boolean Series: True where df1.key is in df2.key
common = set(labeled_seq) & set(labeled_seq)
# same result as above

print("Overlap?" , bool(common))
print("They share:", common)




In [ ]:
nbFolder = 0
nbFrame = 0

for k in final:
    nbFolder +=1
    
    for sub in final[k]:
        nbFrame+=1
        

print('nbFolder',nbFolder)
print('nbFrame',nbFrame)

entry_0 = final[list(final.keys())[0]]

print(entry_0)